完整的可训练模型代码，数据生成和验证代码！


In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from tqdm import tqdm
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用设备: {device}")



# ========== 数据生成 ==========
def generate_parity_data(num_samples=5000, seq_len=16):
    X = torch.randint(0, 2, (num_samples, seq_len)).float().unsqueeze(-1)
    y = (X.squeeze(-1).sum(dim=1) % 2).long()
    return X, y

def generate_mod3_data(num_samples=5000, seq_len=16):
    X = torch.randint(0, 2, (num_samples, seq_len)).float().unsqueeze(-1)
    ones_count = X.squeeze(-1).sum(dim=1)
    y = (ones_count % 3 != 0).long()  # 能被3整除 -> 0，否则 -> 1
    return X, y
def generate_parenthesis_data(num_samples=5000, seq_len=16):
    """
    生成括号匹配任务数据

    输入: [num_samples, seq_len, 1]
        - 0 代表 '('
        - 1 代表 ')'
    标签: [num_samples]
        - 0: 不匹配
        - 1: 匹配
    """
    X = []
    y = []

    for _ in range(num_samples):
        # 随机生成一个长度为 seq_len 的 0/1 序列
        seq = torch.randint(0, 2, (seq_len,)).tolist()

        # 检查括号是否匹配
        balance = 0
        is_valid = True
        for val in seq:
            if val == 0:  # '('
                balance += 1
            else:         # ')'
                balance -= 1
                if balance < 0:  # 出现 ')' 多于 '('
                    is_valid = False
                    break

        # 最终 balance 必须为 0
        if is_valid and balance == 0:
            y.append(1)
        else:
            y.append(0)

        X.append(seq)

    X = torch.tensor(X, dtype=torch.float32).unsqueeze(-1)
    y = torch.tensor(y, dtype=torch.long)

    return X, y


def focal_loss(pred, target, gamma=2.0, alpha=0.25):
    """
    Focal Loss = -alpha * (1 - pt)^gamma * log(pt)

    gamma: 聚焦参数，越大越关注难样本（推荐 2.0）
    alpha: 平衡参数，处理类别不平衡（推荐 0.25）
    """
    ce_loss = F.cross_entropy(pred, target, reduction='none')
    pt = torch.exp(-ce_loss)  # pt: 正确类别的预测概率
    focal = alpha * (1 - pt) ** gamma * ce_loss
    return focal.mean()

def create_dataloaders(X, y, batch_size=64):
    dataset = TensorDataset(X, y)
    return DataLoader(dataset, batch_size=batch_size, shuffle=True), DataLoader(dataset, batch_size=batch_size, shuffle=False)

# ========== 真正的复数 Mamba 层 ==========
class ComplexMambaLayer(nn.Module):
    """
    复数状态空间层（论文版）

    设计原则：
        1. 残差连接：让梯度直接流动
        2. 复数归一化：除以模长，保持相位信息
        3. 无激活函数：保持线性可解释性
    """
    def __init__(self, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim

        # 旋转角度投影
        self.theta_proj = nn.Linear(hidden_dim, 1)

        # 输入映射 B（实部虚部共享）
        self.B_proj = nn.Linear(hidden_dim, hidden_dim)

        # 可学习的衰减因子
        self.log_alpha = nn.Parameter(torch.tensor(0.0))

        self.gate = nn.Parameter(torch.ones(hidden_dim) * 0.5)

        # 初始化
        nn.init.zeros_(self.theta_proj.weight)
        nn.init.zeros_(self.theta_proj.bias)

    def forward(self, h_real_seq, h_imag_seq):
        B, T, H = h_real_seq.shape

        h_real_prev = torch.zeros(B, H, device=h_real_seq.device)
        h_imag_prev = torch.zeros(B, H, device=h_real_seq.device)

        outputs_real, outputs_imag = [], []

        for t in range(T):
            h_real_t = h_real_seq[:, t, :]
            h_imag_t = h_imag_seq[:, t, :]

            # ========== 1. 旋转 ==========
            theta_t = torch.tanh(self.theta_proj(h_real_t)) * math.pi
            cos_t, sin_t = torch.cos(theta_t), torch.sin(theta_t)

            h_real_rot = cos_t * h_real_prev - sin_t * h_imag_prev
            h_imag_rot = sin_t * h_real_prev + cos_t * h_imag_prev

            # ========== 2. 递推 ==========
            alpha = torch.exp(self.log_alpha)
            B_real_t = self.B_proj(h_real_t)
            B_imag_t = self.B_proj(h_imag_t)

            h_real_new = alpha * h_real_rot + B_real_t
            h_imag_new = alpha * h_imag_rot + B_imag_t


            outputs_real.append(h_real_new)
            outputs_imag.append(h_imag_new)

            h_real_prev = h_real_new
            h_imag_prev = h_imag_new


         # 4. 堆叠
        outputs_real = torch.stack(outputs_real, dim=1)  # [B, T, H]
        outputs_imag = torch.stack(outputs_imag, dim=1)  # [B, T, H]

        # 5. ★★★ 可学习门控 ★★★
        # gate: [H] -> 广播到 [B, T, H]
        outputs_real_gated = h_real_seq * torch.sigmoid(self.gate)
        outputs_imag_gated = h_imag_seq * torch.sigmoid(self.gate)

        # 6. ★★★ Skip Connection ★★★
        outputs_real_final = outputs_real_gated + F.silu(outputs_real)
        outputs_imag_final = outputs_imag_gated + F.silu(outputs_imag)

        # 7. ★★★ 复数归一化 ★★★
        magnitude = torch.sqrt(outputs_real_final**2 + outputs_imag_final**2 + 1e-8)
        outputs_real_final = outputs_real_final / magnitude
        outputs_imag_final = outputs_imag_final / magnitude

        return outputs_real_final, outputs_imag_final

# ========== 完整模型 ==========
class ComplexMambaModel(nn.Module):
    def __init__(self, input_dim=1, hidden_dim=64, output_dim=2, num_layers=3):
        super().__init__()
        self.encoder = nn.Linear(input_dim, hidden_dim)
        self.layers = nn.ModuleList([ComplexMambaLayer(hidden_dim) for _ in range(num_layers)])
        self.decoder = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        h = torch.tanh(self.encoder(x))
        h_real = h
        h_imag = torch.zeros_like(h)

        for layer in self.layers:
            h_real, h_imag = layer(h_real, h_imag)

        phase = torch.atan2(h_imag, h_real + 1e-8)
        phase_last = phase[:,-1,:]
        return self.decoder(phase_last)

# ========== 训练 ==========
def train_model(model, train_loader, test_loader, epochs=200, lr=0.001, loss_fn=None):
    """
    通用训练函数

    loss_fn: 损失函数，签名为 loss_fn(pred, target) -> tensor
             如果不传，默认使用 CrossEntropyLoss
    """
    if loss_fn is None:
        loss_fn = nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(\
    optimizer, mode='max', patience=20, factor=0.5)
    losses, accs = [], []
    for epoch in tqdm(range(epochs)):
        model.train()
        total_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            pred = model(X_batch)
            loss = loss_fn(pred, y_batch)
            optimizer.zero_grad()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        losses.append(total_loss / len(train_loader))
        if epoch % 10 == 0:
            acc = evaluate_model_f1(model, test_loader)
            scheduler.step(acc)
            accs.append(acc)
            print(f"Epoch {epoch}, Loss: {losses[-1]:.4f}, Acc_F1: {acc:.4f}")
    return losses, accs

def evaluate_model(model, test_loader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            pred = model(X_batch)
            correct += (pred.argmax(1) == y_batch).sum().item()
    return correct / len(test_loader.dataset)

def evaluate_model_f1(model, test_loader):
    model.eval()
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            pred = model(X_batch)
            pred_class = pred.argmax(1)
            all_preds.extend(pred_class.cpu().numpy())
            all_targets.extend(y_batch.cpu().numpy())

    f1 = f1_score(all_targets, all_preds, average='binary')
    return f1


# ========== 主程序 ==========
if __name__ == "__main__":
    #X, y = generate_parity_data(5000, 16)
    #X, y = generate_mod3_data(5000, 16)
    X,  y = generate_parenthesis_data(10000, 16)
    train_loader, test_loader = create_dataloaders(X, y, 64)
    model = ComplexMambaModel(input_dim=1, hidden_dim=128, output_dim=2, num_layers=3).to(device)
    print(f"参数量: {sum(p.numel() for p in model.parameters()):,}")
    losses, accs = train_model(model, train_loader, test_loader, epochs=300, lr=0.0003, loss_fn=focal_loss)
    print(f"最终准确率: {evaluate_model_f1(model, test_loader):.4f}")

使用设备: cpu
参数量: 50,824


  0%|          | 1/300 [00:16<1:23:55, 16.84s/it]

Epoch 0, Loss: 0.0074, Acc_F1: 0.0000


  4%|▎         | 11/300 [02:37<1:11:51, 14.92s/it]

Epoch 10, Loss: 0.0059, Acc_F1: 0.0000


...

Epoch 280, Loss: 0.0084, Acc_F1: 0.1460


 97%|█████████▋| 291/300 [1:07:56<02:11, 14.59s/it]

Epoch 290, Loss: 0.0009, Acc_F1: 0.9624


100%|██████████| 300/300 [1:09:58<00:00, 14.00s/it]


最终准确率: 0.9775


In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from tqdm import tqdm
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用设备: {device}")



# ========== 数据生成 ==========
def generate_parity_data(num_samples=5000, seq_len=16):
    X = torch.randint(0, 2, (num_samples, seq_len)).float().unsqueeze(-1)
    y = (X.squeeze(-1).sum(dim=1) % 2).long()
    return X, y

def generate_mod3_data(num_samples=5000, seq_len=16):
    X = torch.randint(0, 2, (num_samples, seq_len)).float().unsqueeze(-1)
    ones_count = X.squeeze(-1).sum(dim=1)
    y = (ones_count % 3 != 0).long()  # 能被3整除 -> 0，否则 -> 1
    return X, y
def generate_parenthesis_data(num_samples=5000, seq_len=16):
    """
    生成括号匹配任务数据

    输入: [num_samples, seq_len, 1]
        - 0 代表 '('
        - 1 代表 ')'
    标签: [num_samples]
        - 0: 不匹配
        - 1: 匹配
    """
    X = []
    y = []

    for _ in range(num_samples):
        # 随机生成一个长度为 seq_len 的 0/1 序列
        seq = torch.randint(0, 2, (seq_len,)).tolist()

        # 检查括号是否匹配
        balance = 0
        is_valid = True
        for val in seq:
            if val == 0:  # '('
                balance += 1
            else:         # ')'
                balance -= 1
                if balance < 0:  # 出现 ')' 多于 '('
                    is_valid = False
                    break

        # 最终 balance 必须为 0
        if is_valid and balance == 0:
            y.append(1)
        else:
            y.append(0)

        X.append(seq)

    X = torch.tensor(X, dtype=torch.float32).unsqueeze(-1)
    y = torch.tensor(y, dtype=torch.long)

    return X, y


def focal_loss(pred, target, gamma=2.0, alpha=0.25):
    """
    Focal Loss = -alpha * (1 - pt)^gamma * log(pt)

    gamma: 聚焦参数，越大越关注难样本（推荐 2.0）
    alpha: 平衡参数，处理类别不平衡（推荐 0.25）
    """
    ce_loss = F.cross_entropy(pred, target, reduction='none')
    pt = torch.exp(-ce_loss)  # pt: 正确类别的预测概率
    focal = alpha * (1 - pt) ** gamma * ce_loss
    return focal.mean()

def create_dataloaders(X, y, batch_size=64):
    dataset = TensorDataset(X, y)
    return DataLoader(dataset, batch_size=batch_size, shuffle=True), DataLoader(dataset, batch_size=batch_size, shuffle=False)

# ========== 真正的复数 Mamba 层 ==========
class ComplexMambaLayer(nn.Module):
    """
    复数状态空间层（论文版）

    设计原则：
        1. 残差连接：让梯度直接流动
        2. 复数归一化：除以模长，保持相位信息
        3. 无激活函数：保持线性可解释性
    """
    def __init__(self, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim

        # 旋转角度投影
        self.theta_proj = nn.Linear(hidden_dim, 1)

        # 输入映射 B（实部虚部共享）
        self.B_proj = nn.Linear(hidden_dim, hidden_dim)

        # 可学习的衰减因子
        self.log_alpha = nn.Parameter(torch.tensor(0.0))

        self.gate = nn.Parameter(torch.ones(hidden_dim) * 0.5)

        # 初始化
        nn.init.zeros_(self.theta_proj.weight)
        nn.init.zeros_(self.theta_proj.bias)

    def forward(self, h_real_seq, h_imag_seq):
        B, T, H = h_real_seq.shape

        h_real_prev = torch.zeros(B, H, device=h_real_seq.device)
        h_imag_prev = torch.zeros(B, H, device=h_real_seq.device)

        outputs_real, outputs_imag = [], []

        for t in range(T):
            h_real_t = h_real_seq[:, t, :]
            h_imag_t = h_imag_seq[:, t, :]

            # ========== 1. 旋转 ==========
            theta_t = torch.tanh(self.theta_proj(h_real_t)) * math.pi
            cos_t, sin_t = torch.cos(theta_t), torch.sin(theta_t)

            h_real_rot = cos_t * h_real_prev - sin_t * h_imag_prev
            h_imag_rot = sin_t * h_real_prev + cos_t * h_imag_prev

            # ========== 2. 递推 ==========
            alpha = torch.exp(self.log_alpha)
            B_real_t = self.B_proj(h_real_t)
            B_imag_t = self.B_proj(h_imag_t)

            h_real_new = alpha * h_real_rot + B_real_t
            h_imag_new = alpha * h_imag_rot + B_imag_t


            outputs_real.append(h_real_new)
            outputs_imag.append(h_imag_new)

            h_real_prev = h_real_new
            h_imag_prev = h_imag_new


         # 4. 堆叠
        outputs_real = torch.stack(outputs_real, dim=1)  # [B, T, H]
        outputs_imag = torch.stack(outputs_imag, dim=1)  # [B, T, H]

        # 5. ★★★ 可学习门控 ★★★
        # gate: [H] -> 广播到 [B, T, H]
        outputs_real_gated = h_real_seq * torch.sigmoid(self.gate)
        outputs_imag_gated = h_imag_seq * torch.sigmoid(self.gate)

        # 6. ★★★ Skip Connection ★★★
        outputs_real_final = outputs_real_gated + F.silu(outputs_real)
        outputs_imag_final = outputs_imag_gated + F.silu(outputs_imag)

        # 7. ★★★ 复数归一化 ★★★
        magnitude = torch.sqrt(outputs_real_final**2 + outputs_imag_final**2 + 1e-8)
        outputs_real_final = outputs_real_final / magnitude
        outputs_imag_final = outputs_imag_final / magnitude

        return outputs_real_final, outputs_imag_final

# ========== 完整模型 ==========
class ComplexMambaModel(nn.Module):
    def __init__(self, input_dim=1, hidden_dim=64, output_dim=2, num_layers=3):
        super().__init__()
        self.encoder = nn.Linear(input_dim, hidden_dim)
        self.layers = nn.ModuleList([ComplexMambaLayer(hidden_dim) for _ in range(num_layers)])
        self.decoder = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        h = torch.tanh(self.encoder(x))
        h_real = h
        h_imag = torch.zeros_like(h)

        for layer in self.layers:
            h_real, h_imag = layer(h_real, h_imag)

        phase = torch.atan2(h_imag, h_real + 1e-8)
        phase_last = phase[:,-1,:]
        return self.decoder(phase_last)

# ========== 训练 ==========
def train_model(model, train_loader, test_loader, epochs=200, lr=0.001, loss_fn=None):
    """
    通用训练函数

    loss_fn: 损失函数，签名为 loss_fn(pred, target) -> tensor
             如果不传，默认使用 CrossEntropyLoss
    """
    if loss_fn is None:
        loss_fn = nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(\
    optimizer, mode='max', patience=20, factor=0.5)
    losses, accs = [], []
    for epoch in tqdm(range(epochs)):
        model.train()
        total_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            pred = model(X_batch)
            loss = loss_fn(pred, y_batch)
            optimizer.zero_grad()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        losses.append(total_loss / len(train_loader))
        if epoch % 10 == 0:
            acc = evaluate_model_f1(model, test_loader)
            scheduler.step(acc)
            accs.append(acc)
            print(f"Epoch {epoch}, Loss: {losses[-1]:.4f}, Acc_F1: {acc:.4f}")
    return losses, accs

def evaluate_model(model, test_loader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            pred = model(X_batch)
            correct += (pred.argmax(1) == y_batch).sum().item()
    return correct / len(test_loader.dataset)

def evaluate_model_f1(model, test_loader):
    model.eval()
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            pred = model(X_batch)
            pred_class = pred.argmax(1)
            all_preds.extend(pred_class.cpu().numpy())
            all_targets.extend(y_batch.cpu().numpy())

    f1 = f1_score(all_targets, all_preds, average='binary')
    return f1


# ========== 主程序 ==========
if __name__ == "__main__":
    #X, y = generate_parity_data(5000, 16)
    #X, y = generate_mod3_data(5000, 16)
    X,  y = generate_parenthesis_data(10000, 16)
    train_loader, test_loader = create_dataloaders(X, y, 64)
    model = ComplexMambaModel(input_dim=1, hidden_dim=128, output_dim=2, num_layers=3).to(device)
    print(f"参数量: {sum(p.numel() for p in model.parameters()):,}")
    losses, accs = train_model(model, train_loader, test_loader, epochs=300, lr=0.0003, loss_fn=focal_loss)
    print(f"最终准确率: {evaluate_model_f1(model, test_loader):.4f}")

使用设备: cpu
参数量: 50,824


  0%|          | 1/300 [00:16<1:23:55, 16.84s/it]

Epoch 0, Loss: 0.0074, Acc_F1: 0.0000


  4%|▎         | 11/300 [02:37<1:11:51, 14.92s/it]

Epoch 10, Loss: 0.0059, Acc_F1: 0.0000


...

Epoch 280, Loss: 0.0084, Acc_F1: 0.1460


 97%|█████████▋| 291/300 [1:07:56<02:11, 14.59s/it]

Epoch 290, Loss: 0.0009, Acc_F1: 0.9624


100%|██████████| 300/300 [1:09:58<00:00, 14.00s/it]


最终准确率: 0.9775


# 新增區段

In [12]:
import torch
import random

def generate_parenthesis_data(num_samples=5000, seq_len=16):
    """
    生成括号匹配任务数据

    输入: [num_samples, seq_len, 1]
        - 0 代表 '('
        - 1 代表 ')'
    标签: [num_samples]
        - 0: 不匹配
        - 1: 匹配
    """
    X = []
    y = []

    for _ in range(num_samples):
        # 随机生成一个长度为 seq_len 的 0/1 序列
        seq = torch.randint(0, 2, (seq_len,)).tolist()

        # 检查括号是否匹配
        balance = 0
        is_valid = True
        for val in seq:
            if val == 0:  # '('
                balance += 1
            else:         # ')'
                balance -= 1
                if balance < 0:  # 出现 ')' 多于 '('
                    is_valid = False
                    break

        # 最终 balance 必须为 0
        if is_valid and balance == 0:
            y.append(1)
        else:
            y.append(0)

        X.append(seq)

    X = torch.tensor(X, dtype=torch.float32).unsqueeze(-1)
    y = torch.tensor(y, dtype=torch.long)

    return X, y


# 测试生成
X, y = generate_parenthesis_data(50, 16)
for i in range(50):
    seq = X[i, :, 0].int().tolist()
    # 把 0/1 转成 '(' 和 ')' 方便人类阅读
    seq_str = ''.join(['(' if v == 0 else ')' for v in seq])
    label = "匹配" if y[i] == 1 else "不匹配"
    print(f"序列: {seq_str}, 标签: {label}")

序列: (())()(()(())))), 标签: 不匹配
序列: )))))())((())()), 标签: 不匹配
序列: (()())(()))()))), 标签: 不匹配
序列: )()()()()((()()), 标签: 不匹配
序列: ()((()()())()()), 标签: 匹配
序列: ))()())()()()))(, 标签: 不匹配
序列: )(())()())((())(, 标签: 不匹配
序列: ())))()))())))((, 标签: 不匹配
序列: ))()))))))))))(), 标签: 不匹配
序列: )(())(()()())(((, 标签: 不匹配
序列: )))()())(()((()(, 标签: 不匹配
序列: ()()))()))()(()(, 标签: 不匹配
序列: )((()()(()(())(), 标签: 不匹配
序列: ()()(()(()))))(), 标签: 不匹配
序列: )))()(()((()(()(, 标签: 不匹配
序列: ))))))()))()(((), 标签: 不匹配
序列: ))((((()))())))), 标签: 不匹配
序列: ))()()))()((()((, 标签: 不匹配
序列: (())))())())))(), 标签: 不匹配
序列: )(()(()(()(())((, 标签: 不匹配
序列: ()((((()()(()(((, 标签: 不匹配
序列: ()()((())()))))), 标签: 不匹配
序列: ()))(((()()()()), 标签: 不匹配
序列: (((()()((()()(((, 标签: 不匹配
序列: ()))))))()()(()), 标签: 不匹配
序列: (()()()()(((()((, 标签: 不匹配
序列: )(()((((()()())), 标签: 不匹配
序列: )()()()))))((()(, 标签: 不匹配
序列: ()(()))()(()()(), 标签: 不匹配
序列: (((((()()))()))), 标签: 匹配
序列: ))()(((())))))(), 标签: 不匹配
序列: ()))(((((()(((((, 标签: 不匹配
序列: )()())(((((()()(, 标签: 不匹配
序列: ))(())((